In [ ]:
import json
import pandas as pd

# -------------------------
# CONFIGURACIÓN
# -------------------------
RUTA_JSON = "/content/logs_KX7A_20260220-1944_GrupoB.json"
RUTA_SALIDA_CSV = "/content/final_user_metrics.csv"

# Nombres de llaves esperadas en el JSON
COLUMNA_ACTOR = "nombrecompletodelusuario"
COLUMNA_TIEMPO = "hora"

# Formato del timestamp
FORMATO_TIEMPO = "%d/%m/%y, %H:%M:%S"

# Umbral de inactividad (minutos) para cortar sesiones
UMBRAL_INACTIVIDAD_MIN = 20

# Opcional: minutos nocturnos por alumno (según hora de inicio de sesión)
CALCULAR_NOCTURNO = True
HORA_INICIO_NOCTURNO = 0
HORA_FIN_NOCTURNO = 5  # inclusive

# -------------------------
# UTILIDAD: cargar JSON con posible anidamiento doble
# -------------------------
def cargar_registros_json(ruta_json: str):
    with open(ruta_json, "r", encoding="utf-8") as f:
        data = json.load(f)

    # A veces Moodle/exports vienen como [ [ {...}, {...} ] ]
    if isinstance(data, list) and len(data) == 1 and isinstance(data[0], list):
        return data[0]

    # Caso normal: [ {...}, {...} ]
    if isinstance(data, list):
        return data

    # Caso: {"records":[...]} o similar
    if isinstance(data, dict):
        for k in ("records", "eventos", "logs", "data"):
            if k in data and isinstance(data[k], list):
                return data[k]

    raise ValueError("Estructura JSON no reconocida. Ajusta la función cargar_registros_json().")


def main():
    # 1) Cargar JSON
    registros = cargar_registros_json(RUTA_JSON)
    df = pd.DataFrame(registros)

    # 2) Validaciones mínimas
    if COLUMNA_ACTOR not in df.columns:
        raise SystemExit(f"Falta la columna/llave de actor: '{COLUMNA_ACTOR}'. Columnas: {list(df.columns)}")
    if COLUMNA_TIEMPO not in df.columns:
        raise SystemExit(f"Falta la columna/llave de tiempo: '{COLUMNA_TIEMPO}'. Columnas: {list(df.columns)}")

    print("Filas:", len(df))
    print("Columnas:", list(df.columns))

    # 3) Parsear tiempo
    df["event_time"] = pd.to_datetime(df[COLUMNA_TIEMPO], format=FORMATO_TIEMPO, errors="coerce")

    errores = df["event_time"].isna().sum()
    if errores:
        # No seguimos si no podemos confiar en el tiempo
        raise SystemExit(f"Error: {errores} timestamps no se pudieron parsear. Revisa FORMATO_TIEMPO o datos.")

    print("Rango de tiempo:", df["event_time"].min(), "->", df["event_time"].max())

    # 4) Ordenar por alumno y tiempo
    df = df.sort_values([COLUMNA_ACTOR, "event_time"]).copy()

    # 5) Calcular gaps (min) entre eventos consecutivos por alumno
    df["prev_time"] = df.groupby(COLUMNA_ACTOR)["event_time"].shift(1)
    df["gap_min"] = (df["event_time"] - df["prev_time"]).dt.total_seconds() / 60.0

    # 6) Marcar inicio de nueva sesión
    # nueva sesión si: primer evento (gap NaN) o gap > umbral
    df["new_session"] = df["gap_min"].isna() | (df["gap_min"] > UMBRAL_INACTIVIDAD_MIN)

    # 7) session_id incremental por alumno
    df["session_id"] = df.groupby(COLUMNA_ACTOR)["new_session"].cumsum()

    # 8) Construir tabla de sesiones (inicio, fin, duración)
    sesiones = (
        df.groupby([COLUMNA_ACTOR, "session_id"])
        .agg(
            start=("event_time", "min"),
            end=("event_time", "max"),
            events=("event_time", "size"),
        )
        .reset_index()
    )

    # Duración por sesión (min); si events==1 => start==end => 0
    sesiones["duration_min"] = (sesiones["end"] - sesiones["start"]).dt.total_seconds() / 60.0

    # 9) Métricas finales por alumno
    salida = (
        sesiones.groupby(COLUMNA_ACTOR)
        .agg(
            total_time_minutes=("duration_min", "sum"),
            sessions_count=("session_id", "nunique"),
            avg_session_minutes=("duration_min", "mean"),
            median_session_minutes=("duration_min", "median"),
        )
        .reset_index()
        .rename(columns={COLUMNA_ACTOR: "user"})
    )

    # 10) (Opcional) minutos nocturnos
    if CALCULAR_NOCTURNO:
        mask_noche = sesiones["start"].dt.hour.between(
            HORA_INICIO_NOCTURNO, HORA_FIN_NOCTURNO, inclusive="both"
        )
        nocturno = (
            sesiones.loc[mask_noche]
            .groupby(COLUMNA_ACTOR)["duration_min"]
            .sum()
            .reset_index()
            .rename(columns={COLUMNA_ACTOR: "user", "duration_min": "night_time_minutes"})
        )
        salida = salida.merge(nocturno, on="user", how="left")
        salida["night_time_minutes"] = salida["night_time_minutes"].fillna(0.0)

    # 11) Exportar CSV
    salida.to_csv(RUTA_SALIDA_CSV, index=False)

    print("\nListo.")
    print("Umbral (min):", UMBRAL_INACTIVIDAD_MIN)
    print("Usuarios:", salida.shape[0])
    print("Salida:", RUTA_SALIDA_CSV)
    print("\nPreview:")
    print(salida.head(50).to_string(index=False))


if __name__ == "__main__":
    main()

Filas: 74159
Columnas: ['hora', 'nombrecompletodelusuario', 'usuarioafectado', 'contextodelevento', 'componente', 'nombredelevento', 'descripcin', 'origen', 'direccinip']
Rango de tiempo: 2024-08-16 14:50:51 -> 2026-02-20 19:44:25

Listo.
Umbral (min): 20
Usuarios: 115
Salida: /content/final_user_metrics.csv

Preview:
                                 user  total_time_minutes  sessions_count  avg_session_minutes  median_session_minutes  night_time_minutes
          AARON  RAFAEL AHUMADA TARIN          101.500000              58             1.750000                0.275000            0.000000
                 ADRIAN ROMERO RIVERA          133.916667              53             2.526730                0.316667           14.850000
                   ALAN GARCIA FAVILA          219.916667              73             3.012557                0.300000           20.700000
        ALAN GERARDO GARCIA CERVANTES          232.250000              77             3.016234                0.316667      